# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [23]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [ ]:
!pip install pandas

In [25]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [27]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [ ]:
!pip install langchain_community

In [28]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [29]:
#Replace None by your own value and justify
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0.7)

In [30]:
prompt = ChatPromptTemplate.from_template(
    "Write a short, engaging description of this product: {product}. "
    "Explain its main features, benefits, and who it is suitable for."
)

In [31]:

chain = LLMChain(llm=llm, prompt=prompt)

In [32]:
product = "wireless headphones"

result = chain.run(product)
print(result)

Introducing our sleek Wireless Headphones – the perfect blend of sound quality, comfort, and convenience! Designed for audiophiles and casual listeners alike, these headphones feature cutting-edge Bluetooth technology for seamless connectivity, allowing you to enjoy your favorite tunes without the hassle of tangled wires. 

Experience deep, rich sound with powerful bass and crystal-clear highs, thanks to advanced noise-cancellation technology that immerses you in your music or podcasts. The lightweight, ergonomic design ensures all-day comfort, making them ideal for long commutes, workouts, or relaxing at home.

With up to 20 hours of battery life on a single charge, you can enjoy uninterrupted listening, and the built-in microphone allows for hands-free calls, making them perfect for busy professionals or anyone on the go. 

Whether you're a student, fitness enthusiast, or simply a lover of quality audio, our Wireless Headphones are designed to elevate your listening experience to new

## SimpleSequentialChain

In [33]:
from langchain.chains import SimpleSequentialChain

In [55]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a short description of this product: {product}. "
    "Include its main features and benefits."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [35]:
result = chain_one.run("wireless headphones")
print(result)

Introducing our latest Wireless Headphones, designed for the ultimate audio experience on the go! Featuring advanced Bluetooth technology, these headphones offer seamless connectivity to your devices within a 33-foot range, allowing you to enjoy your favorite music and calls without the hassle of tangled wires. 

Key features include noise cancellation to block out unwanted background sounds, a long-lasting battery life that provides up to 20 hours of continuous playtime, and plush ear cushions for maximum comfort during extended use. With built-in touch controls, you can easily manage your tunes and calls with a simple swipe or tap.

The lightweight and foldable design makes them perfect for travel, while the stylish aesthetic adds a modern touch. Whether you're commuting, working out, or relaxing at home, our Wireless Headphones deliver exceptional sound quality and convenience, enhancing your listening experience anywhere you go. Experience freedom and immersion like never before!


In [37]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
     "Write a short, engaging advertisement based on this "
    "product description: {description}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [39]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [40]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Introducing our state-of-the-art wireless headphones, designed for the ultimate listening experience. Featuring advanced Bluetooth technology, these headphones offer seamless connectivity with a range of up to 30 feet, allowing you to move freely without the hassle of wires. Enjoy superior sound quality with deep bass and crystal-clear highs, perfect for music lovers and audiophiles alike.

Key features include:

- **Noise Cancelling**: Immerse yourself in your favorite tunes without distractions from the outside world.
- **Long Battery Life**: With up to 20 hours of playtime on a single charge, you can enjoy extended listening sessions without interruptions.
- **Comfort Fit Design**: Ergonomically designed ear cushions provide a snug, comfortable fit for all-day wear.
- **Built-in Microphone**: Take calls hands-free and access voice assistants effortlessly.

Experience the freedom of wireless listening with our headphones, perfect for workouts, commutes, or relaxing at home. Enjoy the

"🎧 **Unlock Your Sound Experience!** 🎶\n\nIntroducing our **state-of-the-art wireless headphones**—your new companion for the ultimate audio journey! With **advanced Bluetooth technology**, enjoy seamless connectivity up to **30 feet** away. \n\n🌟 **Feel the Beat**: Immerse yourself with superior sound quality, deep bass, and crystal-clear highs. Perfect for music lovers and audiophiles alike!  \n🔇 **Distraction-Free Listening**: Experience true peace with our **Noise Cancelling** feature. Let your music take center stage!  \n🔋 **All-Day Play**: With up to **20 hours of battery life**, dive into extended listening sessions without a pause.  \n🛋️ **Comfort Redefined**: Enjoy a snug fit with our ergonomically designed ear cushions, perfect for all-day wear.  \n📞 **Hands-Free Calling**: Effortlessly switch to calls with the built-in microphone and access your voice assistant with ease.  \n\nWhether you're hitting the gym, commuting, or unwinding at home, experience the **freedom of wirele

**Repeat the above twice for different products**

## SequentialChain

In [51]:
from langchain.chains import SequentialChain

In [71]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
    "Translate the following product review into English. "
    "Return only the translated review:\n\n{review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="english_review"
                    )


In [72]:
# Prompt: summarize the translated review
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this product review in one short sentence:\n\n"
    "{english_review}"
)

# Chain 2
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
)

In [73]:
result = chain_two.invoke({
    "english_review": "The headphones sound good and are comfortable."
})

print(result["summary"])

The headphones offer good sound quality and comfort.


In [74]:
# Prompt 3: identify the language of the original review
third_prompt = ChatPromptTemplate.from_template(
    "What language is this product review written in? "
    "Reply with only the language name.\n\n"
    "{review}"
)

# Chain 3
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="language"
)

In [76]:
# Prompt 4: write a follow-up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a short and friendly follow-up message to a customer "
    "based on this review summary: {summary}\n\n"
    "Write the message in {language}. "
    "Thank the customer and respond to their feedback."
)

# Chain 4
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="followup_message"
)

In [77]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=["english_review", "summary", "followup_message"],
    verbose=False
)

In [78]:
review = df.Review[5]
overall_chain(review)

{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'english_review': "I find the taste mediocre. The foam doesn't hold, it's weird. I buy the same ones in stores and the taste is much better...  \nOld batch or counterfeit!?",
 'summary': 'The product has a mediocre taste and unstable foam compared to store-bought versions, raising concerns about its authenticity.',
 'followup_message': "Bonjour,\n\nMerci beaucoup d'avoir pris le temps de partager votre avis ! Nous sommes désolés d'apprendre que le goût du produit et la stabilité de la mousse n'ont pas répondu à vos attentes. Votre retour est précieux et nous aide à améliorer nos offres.\n\nN'hésitez pas à nous contacter si vous avez d'autres commentaires ou suggestions. Nous sommes là pour vous aider et nous espérons que vous nous donnerez une autre chance à l'avenir.\n\nCordialement,  \n[L'équipe de votre e

**Repeat the above twice for different products or reviews**

## Router Chain

In [79]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [80]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [81]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [90]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [91]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [92]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [93]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [94]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [96]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [97]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body." A black body is a perfect absorber of all incident radiation, meaning it absorbs all wavelengths of light and does not reflect any. When a black body is heated, it emits radiation across a continuous spectrum of wavelengths.\n\nThe key characteristics of black body radiation include:\n\n1. **Temperature Dependence**: The spectrum of radiation emitted by a black body depends solely on its temperature. As the temperature increases, the peak wavelength of the emitted radiation shifts to shorter wavelengths (this is described by Wien\'s displacement law).\n\n2. **Planck\'s Law**: The distribution of emitted radiation can be described by Planck\'s law, which quantifies the intensity of radiation at different wavelengths for a given temperature. This law was pivotal in the development of quantum mechanics.\n\n3. **Stefan-Boltzmann Law**: The total energy emitted per unit surf

In [98]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'To solve the problem \\(2 + 2\\), we can break it down into its component parts:\n\n1. Identify the numbers involved: We have the numbers 2 and 2.\n2. Understand the operation: The operation we are performing is addition.\n\nNow, we can combine the two numbers:\n\n\\[\n2 + 2 = 4\n\\]\n\nSo, the answer to the question \\(2 + 2\\) is \\(4\\).'

In [99]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA serves as the fundamental blueprint for life. Here are several key reasons why this is the case:\n\n1. **Genetic Information Storage**: DNA (deoxyribonucleic acid) contains the genetic instructions necessary for the development, functioning, growth, and reproduction of all living organisms. It encodes the information required to produce proteins, which perform a vast array of functions within cells.\n\n2. **Cellular Function and Identity**: Each cell in our body has a specific role, whether it be a muscle cell, nerve cell, or skin cell. The DNA in each cell contains the genes that dictate its function and characteristics. While all cells have the same DNA, different genes are expressed in different cell types, leading to cellular differentiation.\n\n3. **Replication and Division**: When cells divide, they must replicate their DNA so that each new cell receives an identical copy of the genetic material. This ensures that the genetic infor

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [100]:
# Test 1: computer science
question_1 = "Explain what a Python list is with a simple example."

print("QUESTION 1:", question_1)
print("ANSWER:")
print(chain.run(question_1))

# Test 2: history
question_2 = "Why was the Berlin Wall built?"

print("\nQUESTION 2:", question_2)
print("ANSWER:")
print(chain.run(question_2))

# Test 3: biology
question_3 = "How do plants use sunlight to make food?"

print("\nQUESTION 3:", question_3)
print("ANSWER:")
print(chain.run(question_3))

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


QUESTION 1: Explain what a Python list is with a simple example.
ANSWER:
computer science: {'input': 'Explain what a Python list is with a simple example.'}

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
A Python list is a built-in data structure that allows you to store an ordered collection of items. Lists can hold a variety of data types, including integers, floats, strings, and even other lists. They are mutable, meaning you can change their content after they have been created (e.g., adding, removing, or modifying elements).

### Key Features of Python Lists:
- **Ordered**: The items in a list maintain their order, and you can access them using their index.
- **Mutable**: You can modify a list after it has been created.
- **Dynamic**: Lists can grow or shrink in size as needed.
- **Heterogeneous**: Lists can contain elements of different data types.

### Creating a List
You can create a list by placing a comma-separated sequence of items inside square brackets `[]`.

### Example
Here's a simple example of a Python list:

```python
# Creating a list of fruits
fruits = ["apple", "banana", "cherry", "date"]

# Accessing elements
print(fruits[0])  # Output: apple
pr

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
The Berlin Wall, erected on August 13, 1961, was a significant symbol of the Cold War and the division between East and West. Its construction was driven by a combination of political, economic, and social factors:

1. **Political Tensions**: After World War II, Germany was divided into East and West, with East Germany (German Democratic Republic, GDR) aligned with the Soviet Union and West Germany (Federal Republic of Germany, FRG) aligned with the West. The ideological conflict between communism and capitalism heightened tensions, leading to a divided Berlin, which became a focal point of Cold War hostilities.

2. **Mass Emigration**: In the years leading up to the Wall's construction, East Germany faced a significant brain drain, as many of its citizens, particularly skilled workers and professionals, fled to West Berlin and then to West Germany. This emigration was a major concern for the East German government, as it undermined the economy and the legitimacy of 